In [ ]:
!pip install statsforecast utilsforecast coreforecast mlforecast lightgbm neuralforecast
!pip install git+https://github.com/amazon-science/chronos-forecasting.git

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from utilsforecast.plotting import plot_series
from utilsforecast.losses import bias, rmse, mae, mape, bias
from utilsforecast.evaluation import evaluate

from statsforecast import StatsForecast
from statsforecast.models import Naive, SeasonalNaive, AutoETS, MSTL, AutoARIMA

import lightgbm as lgb
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean, ExponentiallyWeightedMean

from neuralforecast import NeuralForecast
from neuralforecast.auto import AutoNBEATS, AutoNHITS

import torch
from chronos import ChronosPipeline

In [ ]:
df = pd.read_parquet('/content/sample_data/sample_hotels-1.parquet')
df.info()
plot_series(df, max_ids=19)

In [ ]:
hotels_to_exclude = ['hotel_28', 'hotel_77']
df_clean = (df[~df['unique_id'].isin(hotels_to_exclude)].copy())
print(f"number of hotels: {df_clean['unique_id'].nunique()}")

In [ ]:
print(df_clean['holiday_flag'].value_counts())

In [ ]:
print(df_clean['target_year'].value_counts())

In [ ]:
df_clean = df_clean.drop(columns=['holiday_flag'])
df_clean = df_clean.drop(columns=['target_year'])

In [ ]:
df_clean['location_type'] = df_clean['location_type'].replace('', 'Unknown')

In [ ]:
df_clean.isna().sum()

In [ ]:
df_clean.duplicated().sum()

In [ ]:
columns_to_drop = [f'otb_{i}' for i in range(1, 29)]
df_clean = df_clean.drop(columns=columns_to_drop)
display(df_clean.head())

In [ ]:
df_clean.info()

In [ ]:
df_processed = pd.get_dummies(df_clean, columns=['target_day', 'target_month', 'location_type', 'hotel_type'], drop_first=True, dtype=int)

**TRAIN/TEST SPLIT**

In [ ]:
df_clean['ds'].max()

In [ ]:
cutoff = pd.Timestamp('2023-06-02')

train = df_clean[df_clean['ds'] <= cutoff]
test  = df_clean[df_clean['ds'] >  cutoff]

In [ ]:
train.shape

In [ ]:
test.shape

In [ ]:
df_base = df_clean[['unique_id', 'ds', 'y']]

In [ ]:
train_base = train[['unique_id', 'ds', 'y']]
test_base = test[['unique_id', 'ds', 'y']]

In [ ]:
train_ml = train.copy()
test_ml = test.copy()

**CROSS VALIDATION**

In [ ]:
cutoff_dates = []
max_date = df_base['ds'].max()

In [ ]:
for i in range(5):
    cutoff = max_date - pd.Timedelta(days=28 * (5 - i))
    cutoff_dates.append(cutoff)

In [ ]:
print("Cutoff dates:", cutoff_dates)

In [ ]:
sf_models = [
    Naive(),
    SeasonalNaive(season_length=7),
    AutoETS(season_length=7),
    AutoARIMA(season_length=7)
]

In [ ]:
sf = StatsForecast(
    models=sf_models,
    freq='D',
    n_jobs=-1
)

In [ ]:
cv_base = sf.cross_validation(
    df=df_base,
    h= 28,
    step_size= 28,
    n_windows= 5,
)

In [ ]:
print(f"Statistical models CV shape: {cv_base.shape}")

In [ ]:
cv_base.shape

In [ ]:
lgb_model = MLForecast(
    models={'LightGBM': lgb.LGBMRegressor(verbose=-1)},
    freq='D',
    lags=[7, 14, 21, 28],
    lag_transforms={
        7: [RollingMean(window_size=7), RollingMean(window_size=14)],
        14: [RollingMean(window_size=7)]
    },
    date_features=['dayofweek', 'day', 'month']
)

In [ ]:
cv_lgb = lgb_model.cross_validation(
    df=df_base,
    h=28,
    step_size=28,
    n_windows=5
)

In [ ]:
print(f"LightGBM CV shape: {cv_lgb.shape}")

In [ ]:
cv_lgb.head()

In [ ]:
auto_nbeats = AutoNBEATS(
    h=28,
    num_samples=1,
    cpus=1,
    gpus=0
)

auto_nhits = AutoNHITS(
    h=28,
    num_samples=1,
    cpus=1,
    gpus=0
)

In [ ]:
cv_neural_list = []

In [ ]:
for i, cutoff in enumerate(cutoff_dates):
    print(f"\nProcessing Neural Networks fold {i+1}/5 - Cutoff: {cutoff}")

In [ ]:
train_fold = df_base[df_base['ds'] <= cutoff].copy()
test_fold = (df_base[(df_base['ds'] > cutoff) & (df_base['ds'] <= cutoff + pd.Timedelta(days=28))].copy())

In [ ]:
nf = NeuralForecast(
        models=[auto_nbeats, auto_nhits],
        freq='D'
    )

In [ ]:
nf.fit(df=train_fold)
forecasts = nf.predict()

In [ ]:
forecasts = forecasts.reset_index()
forecasts['cutoff'] = cutoff

In [ ]:
fold_results = (test_fold.merge(forecasts, on=['unique_id', 'ds'], how='left'))

In [ ]:
cv_neural_list.append(fold_results)

In [ ]:
cv_neural = pd.concat(cv_neural_list, ignore_index=True)

In [ ]:
print(f"Neural Networks CV shape: {cv_neural.shape}")

In [ ]:
cv_neural.head()

In [ ]:
chronos_pipeline = ChronosPipeline.from_pretrained("amazon/chronos-t5-small",
    device_map="cuda" if torch.cuda.is_available() else "cpu",
    torch_dtype=torch.bfloat16,
)

In [ ]:
cv_chronos_list = []

In [ ]:
for i, cutoff in enumerate(cutoff_dates):
    print(f"\nProcessing Chronos fold {i+1}/5 - Cutoff: {cutoff}")

In [ ]:
train_fold = df_base[df_base['ds'] <= cutoff].copy()
test_fold = (df_base[(df_base['ds'] > cutoff) & (df_base['ds'] <= cutoff + pd.Timedelta(days=28))].copy())

In [ ]:
unique_hotels = train_fold['unique_id'].unique()
forecasts_list = []

In [ ]:
for hotel in unique_hotels:
    hotel_data = (train_fold[train_fold['unique_id'] == hotel].sort_values('ds'))

In [ ]:
context = torch.tensor(hotel_data['y'].values)

In [ ]:
forecast = chronos_pipeline.predict(context, prediction_length=28, num_samples=20)

In [ ]:
forecast_median = torch.median(forecast, dim=1).values.squeeze().numpy()

In [ ]:
forecast_dates = pd.date_range(
    start=cutoff + pd.Timedelta(days=1),
    periods=28,
    freq='D'
)

In [ ]:
hotel_forecast = pd.DataFrame({
    'unique_id': hotel,
    'ds': forecast_dates,
    'Chronos': forecast_median
})

In [ ]:
forecasts_list.append(hotel_forecast)

In [ ]:
forecasts = pd.concat(forecasts_list, ignore_index=True)
forecasts['cutoff'] = cutoff

In [ ]:
fold_results = (test_fold.merge(forecasts, on=['unique_id', 'ds'], how='left'))

In [ ]:
cv_chronos_list.append(fold_results)

In [ ]:
cv_chronos = pd.concat(cv_chronos_list, ignore_index=True)

In [ ]:
print(f"Chronos CV shape: {cv_chronos.shape}")

In [ ]:
cv_chronos.head()

In [ ]:
cv_all = (
    cv_base
    .merge(
        cv_lgb[['unique_id', 'ds', 'cutoff', 'LightGBM']],
        on=['unique_id', 'ds', 'cutoff'],
        how='left'
    )
    .merge(
        cv_neural[['unique_id', 'ds', 'cutoff', 'AutoNBEATS', 'AutoNHITS']],
        on=['unique_id', 'ds', 'cutoff'],
        how='left'
    )
    .merge(
        cv_chronos[['unique_id', 'ds', 'cutoff', 'Chronos']],
        on=['unique_id', 'ds', 'cutoff'],
        how='left'
    )
)

In [ ]:
print(f"\nCombined CV results shape: {cv_all.shape}")

In [ ]:
cv_all.head()

In [ ]:
model_list = ['Naive', 'SeasonalNaive', 'AutoETS', 'AutoARIMA', 'LightGBM', 'AutoNBEATS', 'AutoNHITS', 'Chronos']

In [ ]:
cv_evaluation = evaluate(
    df=cv_all,
    metrics=[bias, mae, rmse, mape],
    models=model_list
)

In [ ]:
display(cv_evaluation)

In [ ]:
cv_evaluation.to_csv('cv_evaluation_results.csv', index=False)

In [ ]:
per_series_results = []

In [ ]:
for hotel in cv_all['unique_id'].unique():
    hotel_data = cv_all[cv_all['unique_id'] == hotel].copy()

    hotel_eval = evaluate(
        df=hotel_data,
        metrics=[bias, mae, rmse, mape],
        models=model_list
    )

    hotel_eval['unique_id'] = hotel
    per_series_results.append(hotel_eval)

In [ ]:
cv_evaluation_per_series = pd.concat(per_series_results, ignore_index=True)

In [ ]:
display(cv_evaluation_per_series.head(20))

In [ ]:
cv_evaluation_per_series.to_csv('cv_evaluation_per_series.csv', index=False)

In [ ]:
wins_list = []

In [ ]:
for hotel in cv_evaluation_per_series['unique_id'].unique():
    hotel_data = cv_evaluation_per_series[cv_evaluation_per_series['unique_id'] == hotel]

    for metric_name in hotel_data['metric'].unique():
        metric_row = hotel_data[hotel_data['metric'] == metric_name].iloc[0]

        metric_values = {}
        for model in model_list:
            if model in metric_row.index:
                metric_values[model] = metric_row[model]

        if metric_values:
            best_model = min(metric_values, key=metric_values.get)
            wins_list.append({
                'unique_id': hotel,
                'metric': metric_name,
                'winning_model': best_model
            })

In [ ]:
wins_df = pd.DataFrame(wins_list)

In [ ]:
wins_summary = (
    wins_df
    .groupby('winning_model')
    .size()
    .reset_index(name='total_wins')
    .sort_values('total_wins', ascending=False)
)

In [ ]:
display(wins_summary)

In [ ]:
wins_df.to_csv('model_wins_detailed.csv', index=False)
wins_summary.to_csv('model_wins_summary.csv', index=False)

In [ ]:
sf_final = StatsForecast(models=sf_models, freq='D', n_jobs=-1)
sf_final.fit(df=train_base)
forecasts_stat = sf_final.predict(h=28).reset_index()

In [ ]:
lgb_final = MLForecast(
    models={'LightGBM': lgb.LGBMRegressor(verbose=-1, random_state=42)},
    freq='D',
    lags=[7, 14, 21, 28],
    lag_transforms={
        7: [RollingMean(window_size=7), RollingMean(window_size=14)],
        14: [RollingMean(window_size=7)]
    },
    date_features=['dayofweek', 'day', 'month']
)

In [ ]:
lgb_final.fit(df=train_base, static_features=[], dropna=True)

In [ ]:
forecasts_lgb = lgb_final.predict(h=28)

In [ ]:
nf_final = NeuralForecast(models=[auto_nbeats, auto_nhits], freq='D')
nf_final.fit(df=train_base)
forecasts_neural = nf_final.predict().reset_index()

In [ ]:
chronos_forecasts_list = []
unique_hotels_temp = train_base['unique_id'].unique()

In [ ]:
for hotel in unique_hotels_temp:
    hotel_data = (
        train_base[train_base['unique_id'] == hotel]
        .sort_values('ds')
    )

    context = torch.tensor(hotel_data['y'].values, dtype=torch.float32)
    forecast = chronos_pipeline.predict(context, 28)
    forecast_median = torch.median(forecast, dim=1).values.squeeze().numpy()

    forecast_dates = pd.date_range(
        start=train_base['ds'].max() + pd.Timedelta(days=1),
        periods=28,
        freq='D'
    )

    chronos_forecasts_list.append(
        pd.DataFrame({
            'unique_id': hotel,
            'ds': forecast_dates,
            'Chronos': forecast_median
        })
    )

In [ ]:
forecasts_chronos = pd.concat(chronos_forecasts_list, ignore_index=True)

In [ ]:
test_forecasts = (
    test_base
    .merge(forecasts_stat, on=['unique_id', 'ds'], how='left')
    .merge(forecasts_lgb[['unique_id', 'ds', 'LightGBM']], on=['unique_id', 'ds'], how='left')
    .merge(forecasts_neural, on=['unique_id', 'ds'], how='left')
    .merge(forecasts_chronos, on=['unique_id', 'ds'], how='left')
)

In [ ]:
test_forecasts.shape

In [ ]:
test_evaluation = evaluate(
    df=test_forecasts,
    metrics=[bias, mae, rmse, mape],
    models=model_list
)

In [ ]:
display(test_evaluation)

In [ ]:
test_evaluation.to_csv('test_evaluation_results.csv', index=False)

In [ ]:
test_evaluation_per_series_list = []

In [ ]:
for hotel in test_forecasts['unique_id'].unique():
    hotel_data = test_forecasts[test_forecasts['unique_id'] == hotel].copy()

    hotel_eval = evaluate(
        df=hotel_data,
        metrics=[bias, mae, rmse, mape],
        models=model_list
    )

    hotel_eval['unique_id'] = hotel
    test_evaluation_per_series_list.append(hotel_eval)

In [ ]:
test_evaluation_per_series = pd.concat(test_evaluation_per_series_list, ignore_index=True)

In [ ]:
display(test_evaluation_per_series.head(20))

In [ ]:
test_evaluation_per_series.to_csv('test_evaluation_per_series.csv', index=False)

In [ ]:
test_forecasts.to_csv('test_forecasts_all_models.csv', index=False)

In [ ]:
import os
os.makedirs('forecast_plots', exist_ok=True)

In [ ]:
unique_hotels = test_forecasts['unique_id'].unique()

In [ ]:
for hotel in unique_hotels:
    hotel_train = train_base[train_base['unique_id'] == hotel].copy()
    hotel_test = test_forecasts[test_forecasts['unique_id'] == hotel].copy()

    plt.figure(figsize=(14, 6))

    recent_train = hotel_train.tail(28)

    plt.plot(recent_train['ds'], recent_train['y'], label='Training Data')
    plt.plot(hotel_test['ds'], hotel_test['y'], label='Actual', marker='o')

    colors = ['red', 'green', 'orange', 'purple', 'brown', 'pink', 'gray', 'cyan']
    styles = ['-', '--', '-.', ':', '-', '--', '-.', ':']

    for model, color, style in zip(model_list, colors, styles):
        plt.plot(hotel_test['ds'], hotel_test[model], label=model, color=color, linestyle=style, linewidth=2, alpha=0.8)

    y_min = min(hotel_test['y'].min(), recent_train['y'].min()) * 0.9
    y_max = max(hotel_test['y'].max(), recent_train['y'].max()) * 1.1
    plt.ylim(y_min, y_max)

    plt.xlabel('Date')
    plt.ylabel('Occupancy')
    plt.title(f'Forecast vs Actual: {hotel}')
    plt.legend(loc='best')
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(f'forecast_plots/{hotel}_forecast.png', dpi=150, bbox_inches='tight')
    plt.close()